# NextMove — Padel Detector: Transfer Learning on Kaggle GPU
Fine-tunes a COCO-pretrained YOLO11 backbone on a public padel detection
dataset, then exports a Core ML model for the iOS app.

**Pipeline:** COCO-pretrained weights → fix labels → fine-tune → validate →
export to Core ML (`PadelDetector`).

> Enable GPU: Settings → Accelerator → **GPU T4 x2**. Turn **Internet** On.
>
> **Recommended:** run via **Save Version → Save & Run All (Commit)** so the
> session can't idle-timeout mid-training and the artifacts zip is waiting in
> the output when it finishes.
>
> The `ball` class has the fewest boxes (~468), so it is the hardest class;
> this notebook trains at higher resolution with small-object augmentation to
> lift it. Judge success by the per-class **BALL mAP50**, not the overall
> number (which is dominated by easy static classes: wall / net / field).

## 1. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow coremltools
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. Download the padel dataset
Get a free Roboflow API key: https://app.roboflow.com/settings/api

Dataset: **Plaimaker `padel-tkrqs`** (Roboflow Universe, Public Domain).
Object detection, 6 classes: `ball, field, net, outside-field, player, wall`.
The iOS app only uses `ball` and `player`; the rest are court context.

The download is wrapped in a retry loop — Roboflow's export can stall at 50%
on the first request while it generates the export; a retry returns the cached
result.

In [ ]:
import time
from roboflow import Roboflow

ROBOFLOW_API_KEY = ''  # <-- paste your free key
version = Roboflow(api_key=ROBOFLOW_API_KEY).workspace('plaimaker').project('padel-tkrqs').version(1)

dataset = None
for attempt in range(3):
    try:
        dataset = version.download('yolov8', overwrite=True)
        break
    except Exception as e:
        print(f'attempt {attempt+1} failed: {e}; retrying in 15s')
        time.sleep(15)
assert dataset is not None, 'dataset download failed after retries'
print('Dataset at:', dataset.location)

## 2b. Fix the labels (REQUIRED for this dataset)
The Roboflow export mixes **segmentation polygons** with detection boxes in the
same label files. YOLO's detection trainer rejects mixed files and would
silently drop nearly every image. This converts each polygon to a tight
bounding box so every row is a clean detection box.

In [ ]:
from pathlib import Path

def poly_to_bbox(coords):
    xs, ys = coords[0::2], coords[1::2]
    return (min(xs)+max(xs))/2, (min(ys)+max(ys))/2, max(xs)-min(xs), max(ys)-min(ys)
clamp01 = lambda v: min(max(v, 0.0), 1.0)

root = Path(dataset.location)
converted = rows = files = 0
for split in ('train', 'valid', 'test'):
    ldir = root/split/'labels'
    if not ldir.is_dir():
        continue
    for f in ldir.glob('*.txt'):
        out = []
        for line in f.read_text().splitlines():
            p = line.split()
            if not p:
                continue
            cls, nums = p[0], [float(x) for x in p[1:]]
            if len(nums) == 4:
                xc, yc, w, h = nums
            elif len(nums) >= 6 and len(nums) % 2 == 0:
                xc, yc, w, h = poly_to_bbox(nums); converted += 1
            else:
                continue
            xc, yc, w, h = map(clamp01, (xc, yc, w, h))
            if w > 0 and h > 0:
                out.append(f'{cls} {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}')
        f.write_text('\n'.join(out) + ('\n' if out else ''))
        files += 1; rows += len(out)
print(f'Fixed {files} label files, {rows} boxes ({converted} polygons converted).')

## 3. Train (ball-focused)
Higher resolution (960) + small-object augmentation (copy-paste, mixup) +
heavier box-loss weight, all to help the tiny ball. `yolo11s` (small) has more
capacity than nano and is still mobile-friendly. `save_period=10` writes a
checkpoint every 10 epochs so a restart leaves recoverable weights.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolo11s.pt')
model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=60, imgsz=960, batch=8, patience=25,
    optimizer='AdamW', lr0=0.001, lrf=0.01,
    mosaic=1.0, close_mosaic=10, scale=0.5, translate=0.1,
    copy_paste=0.3, mixup=0.1, fliplr=0.5, flipud=0.0, box=9.0,
    device=0, project='runs', name='padel_ball', exist_ok=True,
    save_period=10,
)
print('training done')

## 4. Validate — honest per-class numbers
Locates `best.pt` wherever Ultralytics saved it (the exact path varies by
version, which is why we glob instead of hardcoding). The line that matters is
**BALL mAP50**.

In [ ]:
import glob
from ultralytics import YOLO

cands = glob.glob('/kaggle/working/**/padel_ball/weights/best.pt', recursive=True)
assert cands, 'best.pt not found — did training finish?'
best_path = cands[0]
run_dir = Path(best_path).parents[1]   # .../padel_ball
print('Weights:', best_path)

best = YOLO(best_path)
m = best.val(data=f'{dataset.location}/data.yaml', imgsz=960, plots=True)
names = best.names
ap = {int(i): a for i, a in zip(m.box.ap_class_index, m.box.ap50)}
print(f'\nOverall mAP50={m.box.map50:.3f}  mAP50-95={m.box.map:.3f}  P={m.box.mp:.3f}  R={m.box.mr:.3f}')
for i in sorted(ap):
    print(f'  {names[i]:14s} mAP50={ap[i]:.3f}')
print(f'\n>>> BALL mAP50 = {ap.get(0, 0):.3f}   (the number that matters)')

## 5. Export to Core ML
Export imgsz **must match** the training imgsz (960). `nms=True` bakes in
non-max suppression so the Swift side stays simple.

In [ ]:
best.export(format='coreml', nms=True, imgsz=960)
print('Core ML exported next to best.pt as best.mlpackage')

## 6. Package everything into one zip
Bundles weights + Core ML model (renamed `PadelDetector_v2.mlpackage`) + the
full training run (curves, confusion matrix, results.csv) into a single zip in
`/kaggle/working/`. Download it from the output panel (right sidebar → Data).

In [ ]:
import shutil

out = Path('/kaggle/working/padel_artifacts')
shutil.rmtree(out, ignore_errors=True); out.mkdir(parents=True)

shutil.copy2(best_path, out/'best.pt')
print('added best.pt')

mlpkg = run_dir/'weights/best.mlpackage'
if mlpkg.exists():
    shutil.copytree(mlpkg, out/'PadelDetector_v2.mlpackage')
    print('added PadelDetector_v2.mlpackage')
else:
    print('WARNING: best.mlpackage not found — run the export cell first.')

shutil.copytree(run_dir, out/'training_run', ignore=shutil.ignore_patterns('*.mlpackage'))
print('added training_run/')

z = shutil.make_archive('/kaggle/working/padel_artifacts', 'zip', out)
print(f'\n✅ {z} ({Path(z).stat().st_size/1e6:.1f} MB) — download from the output panel')
print('Then: rename PadelDetector_v2.mlpackage → PadelDetector_v1.mlpackage,')
print('drop into Models/Padel/, add to Xcode Copy Bundle Resources.')